# B1 - Feature engineering

Walk through every feature bundle used by Module B and visualise the
outputs on the training split. Each bundle is a stateless function in
`src/module_b/features.py`; the `REGISTRY` composes them in dependency
order so we can pick a list of bundle names and call `build_features`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

## Load the imputed splits

In [ ]:
from data.loaders import load_split

train = load_split('train')
val = load_split('val')
print('train:', train.shape, train.index.min(), '→', train.index.max())
print('val:  ', val.shape, val.index.min(), '→', val.index.max())
train.head()

## Build the full feature bundle

Six bundles, applied in dependency order:
`calendar` → `lags` → `fundamentals` → `spike` → `regime` → `weather`.
`spike` depends on `fundamentals` (it needs `residual_load`), so the
registry inserts `fundamentals` before `spike` even if we list `spike` first.

In [ ]:
from module_b.features import REGISTRY, build_features

bundles = ('calendar', 'lags', 'fundamentals', 'spike', 'regime', 'weather')
order = REGISTRY.resolve_order(bundles)
print('resolution order:', [b.name for b in order])

feat = build_features(train, bundles=bundles)
print(f'Original columns: {train.shape[1]}  →  After features: {feat.shape[1]}')
added = [c for c in feat.columns if c not in train.columns]
print(f'Added {len(added)} feature columns')

## Calendar features

In [ ]:
calendar_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
                 'month_sin', 'month_cos', 'is_weekend', 'is_holiday_DE',
                 'is_dst_transition']
feat[calendar_cols].iloc[:48].plot(subplots=True, figsize=(11, 10),
                                  layout=(5, 2), sharex=True)
plt.suptitle('Calendar features over the first 48 hours of 2019')
plt.tight_layout()
plt.show()

## Price lags and rolling statistics

Standard EPF lag features: same-hour lags at h-{1,2,3,24,25,48,49,168,169}h
plus 24h / 168h / 336h rolling mean/std/min/max.

In [ ]:
lag_cols = [c for c in feat.columns if c.startswith('price_lag') or c.startswith('price_rmean')]
print('Price lag/roll columns:', lag_cols[:10], '…')

sample = feat.loc['2023-06-01':'2023-06-15', ['price', 'price_lag24h', 'price_lag168h', 'price_rmean24h']]
sample.plot(figsize=(12, 4))
plt.title('Price vs lag-24h, lag-168h, and 24h rolling mean (June 2023)')
plt.ylabel('€/MWh')
plt.tight_layout()
plt.show()

## Fundamentals - residual load and clean spreads

`residual_load = load − wind − solar − run-of-river hydro` is the
single most informative driver of short-term price. `clean_spark_anchor`
and `clean_dark_anchor` proxy the marginal cost of gas-fired and
coal-fired generators after accounting for CO₂.

In [ ]:
win = feat.loc['2023-06-01':'2023-06-08', ['price', 'residual_load',
                                            'clean_spark_anchor', 'renewable_penetration']]
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for ax, col in zip(axes, win.columns):
    ax.plot(win.index, win[col])
    ax.set_ylabel(col)
axes[-1].set_xlabel('Time (UTC)')
plt.suptitle('Price next to residual load, gas-marginal-cost anchor, and renewable share')
plt.tight_layout()
plt.show()

## Spike and regime flags

`is_high_residual_load` fires when residual load is in the top decile
for the same hour of day. `crisis_2022_flag` marks the 2022-03 → 2023-03
European energy crisis - tree models learn separate splits inside vs
outside this regime.

In [ ]:
print('Spike flag activations per year:')
print(feat['is_high_residual_load'].groupby(feat.index.year).mean().round(3))
print('\nCrisis-window rows:', int(feat['crisis_2022_flag'].sum()),
      'of', len(feat))

## Weather features

Five-city weather (Berlin, Cologne, Frankfurt, Hamburg, Munich) ×
4 variables (temp 2m, wind 10m, wind 100m, shortwave radiation). The
weather bundle adds lags at {1, 3, 6, 24}h plus cross-city mean/max
aggregates per variable.

In [ ]:
weather_added = [c for c in feat.columns if
                 (c.startswith('weather_mean_') or c.startswith('weather_max_')
                  or ('_lag' in c and any(c.startswith(city) for city in
                      ('berlin', 'cologne', 'frankfurt', 'hamburg', 'munich'))))]
print(f'{len(weather_added)} weather features added')

win = feat.loc['2023-06-01':'2023-06-08',
               ['weather_mean_wind_speed_100m', 'weather_mean_shortwave_radiation', 'price']]
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(win.index, win['weather_mean_wind_speed_100m'], label='wind 100m (avg)', color='C0')
ax1.plot(win.index, win['weather_mean_shortwave_radiation'] / 50, label='radiation/50', color='C1')
ax1.set_ylabel('weather signal')
ax2 = ax1.twinx()
ax2.plot(win.index, win['price'], color='k', alpha=0.5, label='price')
ax2.set_ylabel('€/MWh')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.title('Weather signals and DA price (week of 2023-06-01)')
plt.tight_layout()
plt.show()

## Prepare the supervised (origin, horizon) layout

Every Module B model consumes one row per (origin timestamp T, horizon h ∈ 1..24).
Past features are sampled at T; forward-known features (calendar, weather forecast)
are sampled at T+h and prefixed with `fut_`.

In [ ]:
import importlib, module_b.features
importlib.reload(module_b.features)
from module_b.features import prepare_supervised, HORIZON_COL

past_cols = ['price_lag1h', 'price_lag24h', 'price_lag168h',
             'price_rmean24h', 'residual_load', 'renewable_penetration',
             'clean_spark_anchor']
future_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
               'is_weekend', 'is_holiday_DE',
               'weather_mean_wind_speed_100m', 'weather_mean_shortwave_radiation']
X, y = prepare_supervised(
    feat,
    horizons=range(1, 25),
    past_cols=past_cols,
    future_cols=future_cols,
)
print(f'X.shape={X.shape}, y.shape={y.shape}')
print(f'Per-horizon rows: {X[HORIZON_COL].value_counts().sort_index().head().to_dict()}…')
X.head()

## Next: B2 - baselines

We now have features + a supervised dataset. Notebook B2 fits the
Lago et al. (2021) baseline suite (naive, seasonal-naive 24h/168h, LEAR)
and reports per-horizon pinball loss / MAE / coverage.